# Redis Vector Backend — Sub-Millisecond Semantic Caching

Medha 0.3.0 ships with `RedisVectorBackend`: a storage backend that uses
**Redis Stack** (RediSearch module) for vector search. Redis is the go-to
choice when you need the **lowest possible search latency** and already run
Redis in your stack.

## Why Redis for vector search?

- **Sub-millisecond P99** — in-memory data structure, no disk I/O on the hot path
- **RediSearch** — production-grade HNSW and FLAT vector indexes built in
- **Zero new service** — if Redis is already in your stack
- **Sentinel / Cluster** — battle-tested HA primitives

This notebook covers:
1. **Standalone HNSW** — `redis_index_algorithm="HNSW"`
2. **Store & search with latency** — measure round-trip time
3. **HNSW vs FLAT** — index parameters and trade-offs
4. **Authentication** — password, SSL, ACL username
5. **Sentinel mode** — high-availability configuration
6. **Latency benchmark** — 100 store + 100 search
7. **When to choose Redis**

**Requirements:**
```bash
pip install "medha-archai[redis,fastembed]"
```

## Prerequisites — Redis Stack via Docker

```bash
docker run -d --name redis-stack \
  -p 6379:6379 -p 8001:8001 \
  redis/redis-stack-server:latest

# Verify RediSearch is available
redis-cli MODULE LIST
```

In [ ]:
import time
import urllib.request

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import RedisVectorBackend
    HAS_REDIS = True
    print("redis package is installed")
except ImportError:
    HAS_REDIS = False
    print("redis package not found — install with: pip install medha-archai[redis]")

import socket
try:
    s = socket.create_connection(("localhost", 6379), timeout=1)
    s.close()
    REDIS_UP = True
    print("Redis reachable at localhost:6379")
except Exception:
    REDIS_UP = False
    print("Redis not reachable — cells requiring Redis will be skipped")

CAN_RUN = HAS_REDIS and REDIS_UP

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: Standalone HNSW — Default Configuration

In [ ]:
if not CAN_RUN:
    print("Skipping — Redis not available.")
else:
    settings = Settings(
        backend_type="redis",
        redis_mode="standalone",
        redis_host="localhost",
        redis_port=6379,
        redis_index_algorithm="HNSW",
    )

    print(f"backend_type          : {settings.backend_type}")
    print(f"redis_mode            : {settings.redis_mode}")
    print(f"redis_index_algorithm : {settings.redis_index_algorithm}")
    print(f"redis_hnsw_m          : {settings.redis_hnsw_m}")

    async with Medha("redis_demo", embedder=embedder, settings=settings) as medha:
        print(f"\nBackend: {type(medha._backend).__name__}")

## Cell 2: Store 5 Entries + Search + Print Latency

In [ ]:
if not CAN_RUN:
    print("Skipping — Redis not available.")
else:
    settings = Settings(
        backend_type="redis",
        redis_mode="standalone",
        redis_index_algorithm="HNSW",
    )

    pairs = [
        ("How many users are there?",      "SELECT COUNT(*) FROM users"),
        ("List active users",              "SELECT * FROM users WHERE status = 'active'"),
        ("Average salary",                 "SELECT AVG(salary) FROM employees"),
        ("Top 10 products by price",       "SELECT * FROM products ORDER BY price DESC LIMIT 10"),
        ("Orders placed today",            "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
    ]

    async with Medha("redis_demo", embedder=embedder, settings=settings) as medha:
        for q, sql in pairs:
            await medha.store(q, sql)

        test_qs = [
            "Total number of users",
            "Show all active members",
            "Something completely unrelated",
        ]

        print(f"  {'Question':<38s}  {'Strategy':<18s}  {'Latency':>8s}")
        print("  " + "-" * 72)
        for q in test_qs:
            t0 = time.perf_counter()
            hit = await medha.search(q)
            elapsed = (time.perf_counter() - t0) * 1000
            print(f"  {q:<38s}  {hit.strategy.value:<18s}  {elapsed:>6.2f}ms")

## Cell 3: HNSW vs FLAT — Index Algorithm Parameters

| Parameter | Description | HNSW default |
|---|---|---|
| `redis_hnsw_m` | Edges per node (higher = better recall, more RAM) | 16 |
| `redis_hnsw_ef_construction` | Build-time search depth | 200 |
| `redis_hnsw_ef_runtime` | Query-time search depth | 10 |

In [ ]:
if not CAN_RUN:
    print("Skipping — Redis not available.")
else:
    # HNSW with custom parameters
    settings_hnsw = Settings(
        backend_type="redis",
        redis_index_algorithm="HNSW",
        redis_hnsw_m=32,                  # more edges → better recall, more RAM
        redis_hnsw_ef_construction=400,   # deeper build → better index quality
        redis_hnsw_ef_runtime=50,         # deeper query → better recall, higher latency
    )
    print("HNSW settings:")
    print(f"  m={settings_hnsw.redis_hnsw_m}  ef_construction={settings_hnsw.redis_hnsw_ef_construction}  ef_runtime={settings_hnsw.redis_hnsw_ef_runtime}")

    # FLAT (exact brute-force) — perfect recall, O(n) latency
    settings_flat = Settings(
        backend_type="redis",
        redis_index_algorithm="FLAT",     # exact search — best for small collections
    )
    print(f"\nFLAT settings:")
    print(f"  algorithm={settings_flat.redis_index_algorithm}  (exact, O(n), best for n < 50k)")

## Cell 4: Authentication — Password, SSL, ACL Username

In [ ]:
# Standalone with password
settings_auth = Settings(
    backend_type="redis",
    redis_host="my-redis.example.com",
    redis_port=6380,
    redis_username="medha_user",    # ACL username (Redis 6+)
    redis_password="secret",        # MEDHA_REDIS_PASSWORD
    redis_ssl=True,
)
print(f"auth: username={settings_auth.redis_username!r}  ssl={settings_auth.redis_ssl}")
print(f"      password set = {settings_auth.redis_password is not None}")

## Cell 5: Sentinel Mode — High Availability

In Sentinel mode, Medha connects to a Redis Sentinel cluster and automatically
follows master failover. Pass the sentinel host:port list and master name.

In [ ]:
# Sentinel mode configuration (placeholder — not connecting)
settings_sentinel = Settings(
    backend_type="redis",
    redis_mode="sentinel",
    redis_sentinel_hosts=[
        "sentinel-1.example.com:26379",
        "sentinel-2.example.com:26379",
        "sentinel-3.example.com:26379",
    ],
    redis_sentinel_master="mymaster",
    redis_password="secret",
)

print(f"redis_mode              : {settings_sentinel.redis_mode}")
print(f"redis_sentinel_master   : {settings_sentinel.redis_sentinel_master}")
print(f"redis_sentinel_hosts    : {settings_sentinel.redis_sentinel_hosts}")
print("(Not connecting — placeholder values)")

## Cell 6: Latency Benchmark — 100 Store + 100 Search

In [ ]:
if not CAN_RUN:
    print("Skipping — Redis not available.")
else:
    N = 100
    settings = Settings(
        backend_type="redis",
        redis_mode="standalone",
        redis_index_algorithm="HNSW",
    )

    async with Medha("redis_bench", embedder=embedder, settings=settings) as medha:
        # Store benchmark
        t0 = time.monotonic()
        for i in range(N):
            await medha.store(
                f"Benchmark question number {i} about table data",
                f"SELECT * FROM table_{i} WHERE active = true",
            )
        store_avg = (time.monotonic() - t0) / N * 1000

        # Clear L1 so searches hit the backend
        await medha.clear_caches()

        # Search benchmark
        t0 = time.monotonic()
        for i in range(N):
            await medha.search(f"Question about entity number {i}")
        search_avg = (time.monotonic() - t0) / N * 1000

        print(f"N={N} operations on Redis (HNSW)")
        print(f"  Avg store  : {store_avg:.2f}ms")
        print(f"  Avg search : {search_avg:.2f}ms")

## When to Choose Redis

| Situation | Recommendation |
|---|---|
| Redis already in your stack | `redis` — zero new service |
| Ultra-low latency required (< 1ms P99) | `redis` — in-memory hot path |
| Dataset fits in RAM (< ~10M entries) | `redis` — ideal range |
| Need HA without extra ops | `redis` sentinel — battle-tested |
| Dataset exceeds RAM | `qdrant` or `elasticsearch` — disk-backed |
| No Redis in stack, pure vector | `qdrant` — lighter footprint |

**Redis shines** when Redis is already in your infrastructure and you need
the lowest possible search latency with a medium-sized dataset (up to ~10M entries).